### Create database 'performance' for the performance indicator and populate it with tables and static records

The demo showcases the story
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-735


## 1 - Initialisation

In [1]:
# Access to Prefect
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  

init_demo()

# Reload the global vars again
from resources.utils import *  

Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000


In [3]:
from importlib import reload
import os
import sys
import prefect
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import rs_workflows

# Local paths
rs_workflows_parent = Path(rs_workflows.__path__[0]).parent

In [4]:
flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
      
  },
}
# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

## 2 - Deploy Prefect flows

We deploy the Prefect workflow that is implemented in the rs-client-libraries git repository to model the database, thus to create the tables and to insert the static records in the `pi_category` table

WARNING: the rs-client-libraries source code must be identical in these 3 environments:

- https://github.com/RS-PYTHON/rs-demo.git
- This Jupyter environment
- The Prefect Docker images

In [5]:
%%bash -s "$rs_workflows_parent"
# Deploy the flow
deploy_file=$(realpath "./init_pi_db_flows.yaml")
echo "Deploying '$deploy_file'..."
(cd $1; prefect --no-prompt deploy --prefect-file "$deploy_file" --all)

Deploying '/home/jovyan/notebooks/sprints/sprint27/init_pi_db_flows.yaml'...
╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'PI db init/PI db init' successfully created with id              │
│ '92bf8e3f-b586-4fdb-9223-3612d681218d'.                                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/92bf8e3f-b586-4fdb-9223-3612d681218d


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'PI db init/PI db init'



In [6]:
# Flow deployment names
pi_deploy = "PI db init/PI db init"
await prefect_utils.wait_for_deployment(pi_deploy)

Finished deploying prefect flow: 'PI db init/PI db init'


## 3 - Run flow

Run the flow to init the pi database.

In [7]:
# Convert to json to trigger prefect flow
params_str = to_json(flow_parameters)

In [8]:
%%bash -s "$pi_deploy" "$params_str"
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 'PI db init/PI db init'...
Created flow run 'curly-trogon'.
└── UUID: 07a2178f-675d-4a23-bbe1-587b2a883486
└── Parameters: {'env': {'owner_id': 'ovidiu'}}
└── Job Variables: {}
└── Scheduled start time: 2025-08-25 08:20:16 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/07a2178f-675d-4a23-bbe1-587b2a883486
Watching flow run 'curly-trogon'...


08:20:20.837 | INFO    | prefect - Flow run is in state 'Pending'
08:20:23.806 | INFO    | prefect - Flow run is in state 'Running'
08:20:24.515 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.
